# 01_cuDF / 02 — cuDF operations

The everyday DataFrame operations, on the GPU, side by side with pandas:

1. Setup and a synthetic retail dataset (orders, customers, products)
2. **GroupBy** — aggregations, multiple keys, `nunique`, `transform`
3. **Joins** — inner, left, semi and anti joins
4. **Sorting** — multi-key sorts, `nlargest`
5. **String operations** — regex search, extract, split, replace
6. **Window functions** — cumulative sums, `shift`, `rank`, rolling windows
7. A small end-to-end pipeline that runs unchanged on pandas and cuDF
8. Timing summary

**Hardware:** NVIDIA T4 (16 GB).

## 1. Setup

In [ ]:
import time, statistics, random
import cudf, cupy as cp, numpy as np, pandas as pd
import matplotlib.pyplot as plt

props = cp.cuda.runtime.getDeviceProperties(0)
print(f"cuDF {cudf.__version__} | pandas {pd.__version__} | GPU {props['name'].decode()}")

In [ ]:
def gpu_sync():
    cp.cuda.Device().synchronize()

def timeit(fn, repeats=3, warmup=1):
    """Median wall time in seconds, with GPU synchronized before each stop."""
    for _ in range(warmup):
        fn(); gpu_sync()
    times = []
    for _ in range(repeats):
        gpu_sync()
        t0 = time.perf_counter()
        fn()
        gpu_sync()
        times.append(time.perf_counter() - t0)
    return statistics.median(times)

timings = {}

def compare(label, pandas_fn, cudf_fn, repeats=3):
    """Time the same operation in pandas and cuDF and record the result."""
    t_pd = timeit(pandas_fn, repeats=repeats, warmup=0)
    t_cu = timeit(cudf_fn, repeats=repeats, warmup=1)
    timings[label] = (t_pd, t_cu)
    print(f"{label:<30} pandas {t_pd * 1e3:9.1f} ms | cuDF {t_cu * 1e3:8.1f} ms | {t_pd / t_cu:6.1f}x")

### The dataset

- `orders` — 5M rows: who bought what, where, when, and how many
- `customers` — 200k rows: segment and city
- `products` — 2,000 rows: category, name and price

A few orders point at product IDs that are **not** in the catalog, so we can see what joins do with unmatched keys.

In [ ]:
N_ORDERS    = 5_000_000
N_CUSTOMERS = 200_000
N_PRODUCTS  = 2_000
rng = cp.random.default_rng(0)

segments = cudf.Series(["Consumer", "Corporate", "Home Office", "Small Business"])
cities   = cudf.Series(["Hyderabad", "Bengaluru", "Pune", "Jaipur", "Chennai", "Delhi", "Mumbai", "Kolkata"])
customers = cudf.DataFrame({
    "customer_id": cp.arange(N_CUSTOMERS, dtype="int32"),
    "segment":     segments.take(rng.integers(0, len(segments), N_CUSTOMERS)).reset_index(drop=True),
    "city":        cities.take(rng.integers(0, len(cities), N_CUSTOMERS)).reset_index(drop=True),
})

random.seed(0)
categories = ["Electronics", "Grocery", "Clothing", "Home", "Sports", "Books"]
colors     = ["Red", "Blue", "Black", "White", "Green"]
sizes      = ["S", "M", "L", "XL"]
prod_cats  = [random.choice(categories) for _ in range(N_PRODUCTS)]
products = cudf.DataFrame({
    "product_id":   np.arange(N_PRODUCTS, dtype="int32"),
    "category":     prod_cats,
    "product_name": [f"{c[:4].upper()}-{i:04d} {random.choice(colors)} {random.choice(sizes)}"
                     for i, c in enumerate(prod_cats)],
    "unit_price":   np.round(np.random.default_rng(0).uniform(5, 500, N_PRODUCTS), 2),
})

start_s = np.datetime64("2026-01-01T00:00:00", "s").astype("int64")
orders = cudf.DataFrame({
    "order_id":    cp.arange(N_ORDERS, dtype="int64"),
    "customer_id": rng.integers(0, N_CUSTOMERS, N_ORDERS).astype("int32"),
    "product_id":  rng.integers(0, N_PRODUCTS + 20, N_ORDERS).astype("int32"),   # +20 = unknown products
    "store_id":    rng.integers(0, 50, N_ORDERS).astype("int16"),
    "quantity":    rng.integers(1, 10, N_ORDERS).astype("int16"),
    "ts":          cudf.Series(start_s + rng.integers(0, 180 * 86_400, N_ORDERS)).astype("datetime64[s]"),
})
orders["month"] = orders["ts"].dt.month

print(f"orders {len(orders):,} | customers {len(customers):,} | products {len(products):,}")
orders.head()

In [ ]:
products.head()

In [ ]:
# Same data in pandas, for comparison
orders_pd    = orders.to_pandas()
customers_pd = customers.to_pandas()
products_pd  = products.to_pandas()

## 2. GroupBy

The split-apply-combine pattern. On the GPU, cuDF hashes or sorts the keys in parallel and aggregates each group with one kernel.

In [ ]:
orders.groupby("store_id")["quantity"].sum().sort_index().head()

In [ ]:
# Several aggregations at once
agg = orders.groupby("store_id").agg({"quantity": ["sum", "mean", "max"], "order_id": "count"})
agg.columns = ["_".join(c) for c in agg.columns]
agg.sort_index().head()

In [ ]:
# Multiple keys
orders.groupby(["store_id", "month"])["quantity"].sum().sort_index().head(8)

In [ ]:
# Distinct customers per store
orders.groupby("store_id")["customer_id"].nunique().sort_index().head()

In [ ]:
# transform() returns one value per *row* (not per group): handy for "compare to the group average"
orders["store_avg_qty"] = orders.groupby("store_id")["quantity"].transform("mean")
orders[["store_id", "quantity", "store_avg_qty"]].head()

In [ ]:
# Same answer as pandas?
g = orders.groupby("store_id")["quantity"].sum().sort_index().to_pandas()
p = orders_pd.groupby("store_id")["quantity"].sum().sort_index()
print("matches pandas:", np.array_equal(g.values, p.values))

In [ ]:
compare("groupby sum (1 key)",
        lambda: orders_pd.groupby("store_id")["quantity"].sum(),
        lambda: orders.groupby("store_id")["quantity"].sum())
compare("groupby multi-agg (2 keys)",
        lambda: orders_pd.groupby(["store_id", "month"]).agg({"quantity": ["sum", "mean"], "order_id": "count"}),
        lambda: orders.groupby(["store_id", "month"]).agg({"quantity": ["sum", "mean"], "order_id": "count"}))
compare("groupby nunique",
        lambda: orders_pd.groupby("store_id")["customer_id"].nunique(),
        lambda: orders.groupby("store_id")["customer_id"].nunique())

## 3. Joins

`merge` works like pandas. Two cuDF differences to know:

- **Row order is not guaranteed.** GPU hash joins write rows in whatever order threads finish. Sort afterwards (or pass `sort=True`) if order matters.
- cuDF has native **semi** and **anti** joins: `how="leftsemi"` keeps left rows that have a match, `how="leftanti"` keeps left rows that don't. Neither adds columns from the right table.

In [ ]:
# Left join: unknown product IDs get <NA> for the product columns
left = orders.merge(products, on="product_id", how="left")
print(f"rows: {len(left):,}  |  orders with no product match: {left['product_name'].isna().sum():,}")
left[left["product_name"].isna()].head(3)

In [ ]:
# Anti join: which orders reference a product that isn't in the catalog?
unknown = orders.merge(products, on="product_id", how="leftanti")
print(f"{len(unknown):,} orders, product IDs:", sorted(unknown["product_id"].unique().to_pandas().tolist())[:10], "...")

In [ ]:
# Semi join: customers that placed at least one order in store 0
store0 = orders[orders["store_id"] == 0][["customer_id"]]
active = customers.merge(store0, on="customer_id", how="leftsemi")
print(f"{len(active):,} of {len(customers):,} customers bought in store 0")

In [ ]:
# The main analysis table: orders + products + customers, with an amount column
sales = (orders.merge(products, on="product_id", how="inner")
               .merge(customers, on="customer_id", how="inner"))
sales["amount"] = sales["quantity"] * sales["unit_price"]

sales_pd = (orders_pd.merge(products_pd, on="product_id", how="inner")
                     .merge(customers_pd, on="customer_id", how="inner"))
sales_pd["amount"] = sales_pd["quantity"] * sales_pd["unit_price"]

print(f"sales: {len(sales):,} rows, {sales.shape[1]} columns")
sales.head()

In [ ]:
compare("join 5M x 2k (inner)",
        lambda: orders_pd.merge(products_pd, on="product_id", how="inner"),
        lambda: orders.merge(products, on="product_id", how="inner"))
compare("join 5M x 200k (inner)",
        lambda: orders_pd.merge(customers_pd, on="customer_id", how="inner"),
        lambda: orders.merge(customers, on="customer_id", how="inner"))
compare("anti join",
        lambda: orders_pd[~orders_pd["product_id"].isin(products_pd["product_id"])],
        lambda: orders.merge(products, on="product_id", how="leftanti"))

## 4. Sorting

In [ ]:
# Multi-key sort, mixed directions
sales.sort_values(["store_id", "amount"], ascending=[True, False])[["store_id", "product_name", "amount"]].head()

In [ ]:
# Top-N without sorting everything
sales.nlargest(5, "amount")[["order_id", "product_name", "quantity", "unit_price", "amount"]]

In [ ]:
compare("sort 1 key (float)",
        lambda: sales_pd.sort_values("amount"),
        lambda: sales.sort_values("amount"))
compare("sort 2 keys",
        lambda: sales_pd.sort_values(["store_id", "amount"], ascending=[True, False]),
        lambda: sales.sort_values(["store_id", "amount"], ascending=[True, False]))
compare("nlargest 10",
        lambda: sales_pd.nlargest(10, "amount"),
        lambda: sales.nlargest(10, "amount"))

## 5. String operations

cuDF stores a string column as one character buffer plus offsets, and runs `.str` methods (including regex) as GPU kernels. This is where pandas, which loops over Python string objects, is usually slowest.

In [ ]:
names = products["product_name"].head(6)
cudf.DataFrame({
    "name":        names,
    "lower":       names.str.lower(),
    "length":      names.str.len(),
    "is_elec":     names.str.startswith("ELEC"),
    "blue_or_red": names.str.contains(r"Blue|Red", regex=True),
    "code":        names.str.extract(r"-(\d{4})")[0].astype("int32"),
    "big_size":    names.str.replace(r" (XL|L)$", " Large", regex=True),
})

In [ ]:
# Split one column into several
parts = names.str.split(" ", expand=True)
parts.columns = ["sku", "color", "size"]
parts

In [ ]:
s_cu = sales["product_name"]
s_pd = sales_pd["product_name"]
compare("str.lower",       lambda: s_pd.str.lower(),                             lambda: s_cu.str.lower(), repeats=1)
compare("str.contains regex", lambda: s_pd.str.contains(r"Blue|Red", regex=True), lambda: s_cu.str.contains(r"Blue|Red", regex=True), repeats=1)
compare("str.extract",     lambda: s_pd.str.extract(r"-(\d{4})"),                lambda: s_cu.str.extract(r"-(\d{4})"), repeats=1)
compare("str.split expand", lambda: s_pd.str.split(" ", expand=True),            lambda: s_cu.str.split(" ", expand=True), repeats=1)

## 6. Window functions

Window functions compute a value for each row from the rows around it: running totals, previous values, ranks, moving averages. Most are written as `groupby(...)` followed by a cumulative or shift operation.

We sort by customer and time first so "previous" and "running" mean what we expect.

In [ ]:
sales = sales.sort_values(["customer_id", "ts"]).reset_index(drop=True)
sales_pd = sales_pd.sort_values(["customer_id", "ts"]).reset_index(drop=True)

by_cust = sales.groupby("customer_id")
sales["order_no"]       = by_cust.cumcount() + 1                 # 1st, 2nd, 3rd order...
sales["running_spend"]  = by_cust["amount"].cumsum()             # lifetime spend so far
sales["prev_amount"]    = by_cust["amount"].shift(1)             # previous order's amount
sales["since_prev"]     = sales["ts"] - by_cust["ts"].shift(1)   # time since previous order

sales[["customer_id", "ts", "amount", "order_no", "running_spend", "prev_amount", "since_prev"]].head(8)

In [ ]:
# Rank each sale within its category (1 = biggest)
sales["rank_in_cat"] = sales.groupby("category")["amount"].rank(method="dense", ascending=False)
sales.sort_values(["category", "rank_in_cat"])[["category", "product_name", "amount", "rank_in_cat"]].head(6)

In [ ]:
# Rolling window: daily revenue and its 7-day moving average
daily = sales.groupby(sales["ts"].dt.floor("D"))["amount"].sum().sort_index()
daily_7d = daily.rolling(7).mean()

ax = daily.to_pandas().plot(figsize=(9, 3), alpha=0.4, label="daily revenue")
daily_7d.to_pandas().plot(ax=ax, label="7-day moving average")
ax.set_ylabel("amount"); ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Rolling inside groups: average of each customer's last 3 orders
sales.groupby("customer_id")["amount"].rolling(3).mean().head(8)

In [ ]:
by_cust_pd = sales_pd.groupby("customer_id")
compare("groupby cumsum",     lambda: by_cust_pd["amount"].cumsum(),   lambda: by_cust["amount"].cumsum())
compare("groupby shift",      lambda: by_cust_pd["amount"].shift(1),   lambda: by_cust["amount"].shift(1))
compare("groupby rank",
        lambda: sales_pd.groupby("category")["amount"].rank(method="dense", ascending=False),
        lambda: sales.groupby("category")["amount"].rank(method="dense", ascending=False))
compare("groupby rolling(3) mean",
        lambda: by_cust_pd["amount"].rolling(3).mean(),
        lambda: by_cust["amount"].rolling(3).mean(), repeats=1)

## 7. One pipeline, two libraries

Because the APIs match, the same function runs on a pandas or a cuDF DataFrame. Question: **top 3 products per category by revenue in March**.

In [ ]:
def top_products(df, month=3, n=3):
    m = df[df["ts"].dt.month == month]
    rev = m.groupby(["category", "product_name"], as_index=False)["amount"].sum()
    rev["rank"] = rev.groupby("category")["amount"].rank(method="first", ascending=False)
    return rev[rev["rank"] <= n].sort_values(["category", "rank"]).reset_index(drop=True)

top_products(sales)

In [ ]:
same = np.allclose(top_products(sales)["amount"].to_numpy(), top_products(sales_pd)["amount"].to_numpy())
print("same result as pandas:", same)
compare("pipeline: top 3 per category", lambda: top_products(sales_pd), lambda: top_products(sales))

## 8. Timing summary

In [ ]:
summary = pd.DataFrame(timings, index=["pandas_s", "cudf_s"]).T
summary["speedup"] = summary["pandas_s"] / summary["cudf_s"]
summary = summary.sort_values("speedup")

ax = summary["speedup"].plot.barh(figsize=(8, 0.35 * len(summary) + 1), color="#4C9F70")
ax.axvline(1, color="gray", lw=1, ls="--")
ax.set_xscale("log")
ax.set_xlabel("speedup vs pandas (log scale, higher is better)")
ax.set_title(f"cuDF vs pandas on {props['name'].decode()}")
for i, v in enumerate(summary["speedup"]):
    ax.text(v, i, f" {v:.1f}x", va="center", fontsize=8)
plt.tight_layout(); plt.show()

summary.round(4)

In [ ]:
import gc
del orders_pd, customers_pd, products_pd, sales_pd, by_cust_pd
gc.collect()
free_b, total_b = cp.cuda.runtime.memGetInfo()
print(f"GPU memory used: {(total_b - free_b) / 1e9:.2f} GB / {total_b / 1e9:.1f} GB")

## Summary

- **GroupBy**: `.agg({...})`, multiple keys, `nunique` and `transform` work as in pandas.
- **Joins**: same `merge` API, plus `leftsemi` / `leftanti`. Output row order is not guaranteed; sort if it matters.
- **Sorting**: `sort_values` with mixed directions and `nlargest` for top-N.
- **Strings**: regex `contains`, `extract`, `replace` and `split(expand=True)` all run on the GPU. This is often the biggest speedup.
- **Windows**: `cumcount`, `cumsum`, `shift`, `rank` and `rolling` inside `groupby`.
- The same function can run on pandas or cuDF: that idea is taken all the way in the next notebook.

**Try it:** set `N_ORDERS = 20_000_000` and rerun. Which operations gain the most?

**Next:** `03_cudf_pandas_accelerator.ipynb` — run unchanged pandas code on the GPU with `cudf.pandas`.